<div align="center">

# fse 101

Turn sentences into vectors, find similar text, and save your results.

</div>

## When do you need this?

You want to find related notes even when they use different words.
You have a list of messages to compare, or a file too large to encode in one call.
**Start here** before trying the other recipes.

## The story in pictures

<picture>
<source media="(prefers-color-scheme: dark)" srcset="media/00-fse-101-dark.svg">
<img src="media/00-fse-101-light.svg" alt="Sentences enter a model. Each becomes a row of numbers. Compare rows to find related text.">
</picture>

A **sentence vector** is a list of numbers that represents a text.
Similar texts often have vectors that point in similar directions.
An **encoder** is the model that turns text into those vectors.
This recipe takes about 15 minutes to read, plus installation and download time.

## 1. Install fse

From a checkout of the development branch, run these commands in a terminal.
Use standard Python 3.13, rather than a free-threaded build.
The [recipe index](README.md#run-a-recipe) explains the shared setup.

```sh
UV_PYTHON=3.13 make install
uv pip install --python .venv/bin/python -r recipes/requirements.txt
```

Open this recipe with that environment's Python kernel.

## 2. Load a real model

POTION-base-8M is a small English **static model**, with a fixed vector for each word or word piece.
The first load downloads the model from the Hugging Face Hub.
The helper only manages temporary files and disables third-party telemetry.

In [ ]:
from _helpers import Recipe
import fse
import numpy as np

recipe = Recipe()
work = recipe.work

In [ ]:
import fse
from _helpers import POTION, POTION_REVISION

encoder = fse.Encoder.from_pretrained(
    POTION, revision=POTION_REVISION, cache_dir=work / "hub", token=False,
)
print("Numbers in each vector:", encoder.output_dim)

## 3. Encode a few sentences

Give the encoder a list of texts.
The result is a NumPy matrix, a table with **one row per text**.

In [ ]:
texts = [
    "A cat is sleeping on the sofa.",
    "A kitten is resting on the couch.",
    "The train leaves the station at noon.",
    "A chef is preparing dinner in the kitchen.",
]
vectors = encoder.encode(texts, workers=1)
print("Matrix shape:", vectors.shape)
print("First five numbers:", vectors[0, :5].round(3))

## 4. Find the most similar sentence

Encode a search phrase, then compare it with every row.
These vectors have unit length, so their dot product gives **cosine similarity**, a score for how closely their directions match.
A higher score means more similar directions, not a probability.

In [ ]:
import numpy as np

query = encoder.encode(["An animal is taking a nap indoors."], workers=1)[0]
scores = vectors @ query
for index in np.argsort(-scores):
    print(f"{scores[index]:.3f}  {texts[index]}")

## 5. Compare two sentences

The same calculation compares any pair of rows.
**Check examples from your own task** before relying on a model's similarity scores.

In [ ]:
related = float(vectors[0] @ vectors[1])
unrelated = float(vectors[0] @ vectors[2])
print(f"Cat and kitten: {related:.3f}")
print(f"Cat and train:  {unrelated:.3f}")
assert related > unrelated

## 6. Encode a file or many texts

For a list that fits in memory, `encode` returns all vectors together.
For a larger source, `encode_batches` accepts small lists and yields one result at a time.
Here we make a UTF-8 file from **12 real AG News articles**, then read three lines per batch.

In [ ]:
from itertools import islice

articles = recipe.news(12)
source = work / "news.txt"
source.write_text("\n".join(t.replace("\n", " ") for t in articles) + "\n", encoding="utf-8")

def file_batches(path, size=3):
    with path.open(encoding="utf-8") as stream:
        while lines := list(islice(stream, size)):
            yield [line.rstrip("\n") for line in lines]

for batch in encoder.encode_batches(file_batches(source), workers=1):
    print("First row:", batch.first_row, "Batch shape:", batch.rows.shape)

## 7. Save the vectors and open them again

A **sink** receives vectors as fse produces them.
`ShardSink` saves them in separate files, called shards, rather than keeping the full matrix in memory.
Keep this store on a supported local filesystem and use one writer.

In [ ]:
store_path = work / "news-vectors"
with fse.ShardSink(store_path, encoder, shard_rows=6) as sink:
    for batch in encoder.encode_batches(file_batches(source), sink=sink, workers=1):
        print("Rows processed in this batch:", batch.n_rows)

with fse.VectorStore.open(store_path) as store:
    saved = store.get_vectors(start=0, stop=3)
    print("Saved rows:", len(store), "First three vectors:", saved.shape)
np.testing.assert_array_equal(saved, encoder.encode(articles[:3], workers=1))

## Result

You have made vectors, ranked related sentences, and reopened a saved store.
**The saved rows match the original encoder output**, which the assertion checks exactly.
The helper deletes this example store at cleanup, so choose your own permanent path when adapting the recipe.

## What next?

- [Continue a file job after an interruption](03-files-with-resume.ipynb), or [train a classifier](02-linear-heads.ipynb).
- [fse API and design](../docs/architecture/runtime.md)
- [Model card and license](https://huggingface.co/minishlab/potion-base-8M)

Back to the [recipe index](README.md) or the [main README](../README.md#recipes-from-a-model-to-a-working-application).

## Clean up

The helper keeps generated files in a temporary folder. Run this after you finish exploring the results.

In [ ]:
recipe.close()